In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_27_try_1.pickle

In [3]:
%%RecordEvent
%%time
### cell 28 ###

# Cleaning column names
question_orig_2021 = 'Which of the following hosted notebook products do you use on a regular basis?'
question_new = 'Do you use any of the following hosted notebook products?'

# 2021 and 2020
df_2021_clean = responses_df_2021.copy()
df_2021_clean.columns = df_2021_clean.columns.str.replace(question_orig_2021, question_new, regex=False)

df_2020_clean = responses_df_2020.copy()
df_2020_clean.columns = df_2020_clean.columns.str.replace(question_orig_2021, question_new, regex=False)

# 2019
colab_pat_2019 = 'Google Colab '
kaggle_pat_2019 = 'Kaggle Notebooks (Kernels) '
question_orig_2019 = question_orig_2021

df_2019_clean = responses_df_2019_cell10.copy()
df_2019_clean.columns = (
    df_2019_clean.columns
        .str.replace(colab_pat_2019, 'Colab Notebooks', regex=False)
        .str.replace(kaggle_pat_2019, 'Kaggle Notebooks', regex=False)
        .str.replace(question_orig_2019, question_new, regex=False)
)

# 2018
question_orig_2018 = 'Which of the following hosted notebooks have you used at work or school in the last 5 years?'
colab_pat_2018 = 'Google Colab'
kaggle_pat_2018 = 'Kaggle Kernels'

df_2018_clean = responses_df_2018_cell10.copy()
df_2018_clean.columns = (
    df_2018_clean.columns
        .str.replace(colab_pat_2018, 'Colab Notebooks', regex=False)
        .str.replace(kaggle_pat_2018, 'Kaggle Notebooks', regex=False)
        .str.replace(question_orig_2018, question_new, regex=False)
)

# Optimized core functions
def grab_subset_of_data_40(original_df, question_of_interest):
    # select all columns containing the question text
    cols = [c for c in original_df.columns if question_of_interest in c]
    # derive simple names as text after the last dash
    new_names = [c.rsplit('-', 1)[-1].lstrip() for c in cols]
    df = original_df.loc[:, cols].rename(columns=dict(zip(cols, new_names)))
    # drop rows where all responses for this question are NaN
    return df.dropna(how='all', subset=new_names)


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_40(
        question_of_interest, include_2017=False):
    # chronological mapping
    mapping = [
        ('2018', df_2018_clean),
        ('2019', df_2019_clean),
        ('2020', df_2020_clean),
        ('2021', df_2021_clean),
        ('2022', responses_df_2022_cell10),
    ]
    if include_2017 and 'responses_df_2017' in globals():
        mapping.insert(0, ('2017', responses_df_2017.copy()))
    dfs = []
    for year, df_src in mapping:
        df_sub = grab_subset_of_data_40(df_src, question_of_interest)
        df_sub['year'] = year
        dfs.append(df_sub)
    combined = pd.concat(dfs, ignore_index=True)
    counts = combined.groupby('year').count().reset_index()
    return combined, counts


def convert_df_of_counts_to_percentages_40(df, df_counts):
    # compute the number of respondents per year
    sizes = df['year'].value_counts().reindex(df_counts['year']).astype(int).values
    count_cols = [c for c in df_counts.columns if c != 'year']
    pct = df_counts.copy()
    # vectorized percentage calculation
    pct.loc[:, count_cols] = pct[count_cols].div(sizes, axis=0).mul(100)
    return pct

# Execute pipeline
question_of_interest_cell40 = question_new
notebooks_df_combined, notebooks_df_combined_counts = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_40(
        question_of_interest_cell40)
)
notebooks_df_combined_percentages = convert_df_of_counts_to_percentages_40(
    notebooks_df_combined, notebooks_df_combined_counts
)

# select and reshape for plotting
notebooks_df_combined_percentages_cell40 = notebooks_df_combined_percentages.loc[
    :, ['year', 'None', 'Kaggle Notebooks', 'Colab Notebooks']
]
df_cell40 = notebooks_df_combined_percentages_cell40.melt(
    id_vars=['year'], value_vars=['None', 'Kaggle Notebooks', 'Colab Notebooks']
)
df_cell40 = df_cell40.rename(columns={'variable': ''})
df_cell40.info()

/opt/conda/envs/elastic/lib/python3.10/site-packages/IPython/core/magics/execution.py:1316: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[55.06108202 50.05405405 50.4091653  53.43082115 56.60377358]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  exec(code, glob, local_ns)
/opt/conda/envs/elastic/lib/python3.10/site-packages/IPython/core/magics/execution.py:1316: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[34.55497382 46.48648649 51.96399345 54.78065242 65.60232221]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  exec(code, glob, local_ns)
/opt/conda/envs/elastic/lib/python3.10/site-packages/IPython/core/magics/execution.py:1316: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[11.16928447

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   year    15 non-null     object 
 1           15 non-null     object 
 2   value   15 non-null     float64
dtypes: float64(1), object(2)
memory usage: 488.0+ bytes
CPU times: user 89.3 ms, sys: 7.68 ms, total: 97 ms
Wall time: 96.9 ms


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_28_try_1.pickle